# Universidad Libre - Seccional Cali<br>Facultad de Ingeniería - Diplomado en Ciencia de Datos<br>(ↄ) Juan Sebastian Diaz Campos, 2026

# 01_consolidar
Plantilla para el desarrollo del proyecto del diplomado de Ciencia de Datos, aplicando buenas prácticas.

---

Este cuaderno se enfoca en la integración de las distintas fuentes de datos en un formato cohesivo y estructurado. Aquí transformamos múltiples conjuntos de datos en una base unificada que servirá para los análisis posteriores.

**Propósito:** Crear una vista unificada y coherente de todos los datos recolectados, facilitando su posterior procesamiento y análisis.

**Tareas habituales:**
- Renombrar archivos
- Unión vertical de archivos complementarios (`union`)
- Combinar archivos (`joins`: inner, left, right, full outer)
- Estandarización inicial de formatos de columnas
- Verificación de consistencia en las uniones
- Validación de cardinalidad en las relaciones
- Gestión de duplicados producto de las uniones

In [2]:
# Librerías a utilizar en este módulo
import os
import pandas as pd
import openpyxl
import glob

In [3]:
cwd = os.getcwd() # current working directory
raw_dir = cwd + '/../data/raw/'
landing_dir = cwd + '/../data/landing/'

## 1. Recorriendo los diferentes directorios y sacando los nombres de los archivos:

In [4]:
archivos = []
for ruta in os.listdir(raw_dir):
    if os.path.isdir(raw_dir + ruta):
        for archivo in os.listdir(raw_dir + ruta):
            # validar si es Excel?
            archivos.append(raw_dir + ruta + '/' + archivo)

archivos

['C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw/aplicaciones/.ipynb_checkpoints',
 'C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw/aplicaciones/report_users_profile_accesclean_2026_05.csv',
 'C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw/aplicaciones/report_users_profile_accesclean_2026_06.csv',
 'C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw/matrices/Matriz Roles y Perfiles AdminBusiness V8.xlsx',
 'C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw/rrhh/1 - HAM_24_Acuerdo_Empleados_Activos.xlsx',
 'C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw/rrhh/Colaboradores Migrados a IDM.xlsx',
 'C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw/solicitudes/solicitudes_jsdiaz_2025.csv',
 'C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw/solicitudes/solicitudes_jsdiaz_2026.csv',
 'C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw/solicitudes/solicitudes_jzamorano_2024.cs

## 2. Separando los diferentes archivos para consolidar cada proceso:

In [5]:
archivo_matriz = None
archivo_empleados_idm = None
archivo_empleados_rrhh = None
archivos_solicitudes = None
archivo_aplicacion = None

for archivo in archivos:
    nombre = archivo.lower()

    if "matriz" in nombre:
        archivo_matriz = archivo
    elif "idm" in nombre:
        archivo_empleados_idm = archivo
    elif "ham" in nombre:
        archivo_empleados_rrhh = archivo
    elif "solicitud" in nombre:
        archivos_solicitudes = archivo
    elif "report" in nombre:
        archivo_aplicacion = archivo

## 3. Inspeccionando los archivos de matrices debido a sus múltiples hojas:

In [6]:
xls = pd.ExcelFile(archivo_matriz)
xls.sheet_names

['Roles VS Cargos', 'Opciones VS Roles']

In [7]:
for hoja in xls.sheet_names:
    df = pd.read_excel(archivo_matriz,sheet_name=hoja,header=None)
    print(hoja)
    display(df.head(15))

Roles VS Cargos


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Matriz de Roles VS Cargos - AdminBusiness,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NRO. MATRIZ,PERFIL,COD PERFIL,TIPO ENTIDAD,COD CARGO,CARGO,COD DIVISION,DIVISION,COD AREA,AREA,ESTADO,OBSERVACION
4,NaN,V08,ADMINISTRADOR,101,Interno,1000066,INSPECTOR DE CIERRE,1300000,OPERACIONES,1300005,PRODUCCIÓN,Activo,NaN
5,NaN,V08,ADMINISTRADOR,101,Interno,1000061,SUPERVISOR DE PRODUCCIÓN,1300000,OPERACIONES,1300005,PRODUCCIÓN,Activo,NaN
6,NaN,V08,APROBADOR GENERAL,102,Interno,1000089,DIRECTOR GENERAL,1000001,PRESIDENCIA_G1,1000000,DIRECCION GENERAL,Activo,NaN
7,NaN,V08,REVISORIA GLOBAL,103,Interno,1000014,ANALISTA DE COMPRAS,1200000,COMPRAS Y MANTENIMIENTO,1200002,ABASTECIMIENTO,Activo,NaN
8,NaN,V08,REVISORIA GLOBAL,103,Interno,1000085,ANALISTA DE ALMACEN,1300000,OPERACIONES,1300002,LOGÍSTICA Y ALMACENES,Activo,NaN
9,NaN,V08,REVISORIA GLOBAL,103,Interno,1000048,ASISTENTE ADMINISTRATIVO,1100000,SERVICIO AL CLIENTE,1100002,ADMINISTRACIÓN Y SOPORTE,Activo,Incluido reciente por solicitud jefe inmediato


Opciones VS Roles


,0,1,2,3,4,5,6,7,8,9,10
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Matriz de Opciones VS Roles - AdminBusiness,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,COD OPCION,OPCION,DETALLE DE LA OPCION,OPCION SENSIBLE,ADMINISTRADOR,APROBADOR GENERAL,REVISORIA GLOBAL,AUDITORIA,SOPORTE_TI,PREP_FLUJOS_ADM,LANZADOR COMPRAS
4,AB-100,Menú Administración,NaN,NaN,X,X,NaN,NaN,X,NaN,NaN
5,AB-101,Consulta Usuarios,Consultar información de usuarios registrados,NaN,X,NaN,NaN,NaN,NaN,NaN,NaN
6,AB-102,Modificar Usuarios,Actualizar usuarios existentes,NaN,X,NaN,NaN,NaN,NaN,NaN,NaN
7,AB-103,Asignar Licencia,Asignar licencias de uso a usuarios,NaN,X,NaN,NaN,NaN,NaN,NaN,NaN
8,AB-104,Cambio Clave,Restablecer o cambiar contraseña,NaN,X,NaN,NaN,NaN,NaN,NaN,NaN
9,AB-105,Crear Festivos,Administrar calendario de días festivos,X,NaN,NaN,NaN,NaN,X,NaN,NaN


## 4. Consolidando los archivos de matrices según su información y proceso a ejecutar:

In [8]:
matriz_roles = pd.read_excel(archivo_matriz, sheet_name="Opciones VS Roles", header=3)
matriz_roles.columns

Index(['COD OPCION', 'OPCION', 'DETALLE DE LA OPCION', 'OPCION SENSIBLE',
       'ADMINISTRADOR', 'APROBADOR GENERAL', 'REVISORIA GLOBAL', 'AUDITORIA',
       'SOPORTE_TI', 'PREP_FLUJOS_ADM', 'LANZADOR COMPRAS'],
      dtype='str')

In [9]:
matriz_cargos = pd.read_excel(archivo_matriz, sheet_name="Roles VS Cargos", header=3)
matriz_cargos.columns

Index(['Unnamed: 0', 'NRO. MATRIZ', 'PERFIL', 'COD PERFIL', 'TIPO ENTIDAD',
       'COD CARGO', 'CARGO', 'COD DIVISION', 'DIVISION', 'COD AREA', 'AREA',
       'ESTADO', 'OBSERVACION'],
      dtype='str')

In [10]:
matriz_cargos.to_csv(landing_dir+'matriz_con_cargos.csv',index=False)
matriz_roles.to_csv(landing_dir+'matriz_con_roles.csv',index=False)

## 5. Consolidando los archivos de recursos humanos según la información requerida:

In [11]:
archivos_rrhh = glob.glob(raw_dir + 'rrhh' + '*/*.xlsx')
archivos_rrhh

['C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw\\rrhh\\1 - HAM_24_Acuerdo_Empleados_Activos.xlsx',
 'C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw\\rrhh\\Colaboradores Migrados a IDM.xlsx']

In [12]:
datos_rrhh = []
for archivo in sorted(archivos_rrhh):
    datos_rrhh.append(pd.read_excel(archivo))

df = pd.concat(datos_rrhh, ignore_index=True)
df

,ID de usuario,Colombia Cedula Ciudadania ID nacional,Colombia Tarjeta de Identidad ID nacional,Colombia Cedula Extranjeria ID nacional,Colombia Cedula Extranjeria 2 ID nacional,Colombia Pasaporte ID nacional,Colombia Cedula Ciudadania 2 ID nacional,Colombia Registro Civil Nacimiento ID nacional,Colombia Cedula Ciudadania 3 ID nacional,Colombia Cedula Ciudadania 4 ID nacional,...,CIUDAD,USUARIO_DE_RED,ESTADO_EMPLEADO,MOTIVO_AUSENTISMO,FECHA_FIN,TIPO_CONTRATO,COD_OFICINA,FECHA_CONTRATACION,FECHA_INICIO_AUSENTISMO,FECHA_FIN_AUSENTISMO
0,20091101.0,16421058.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
1,20091102.0,16490884.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
2,20091103.0,17426686.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
3,20091104.0,17451011.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
4,20091105.0,43418954.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1283,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,SAN JOSE DE TICLLAS,MZEÑA2,ACTIVO,NaN,NaN,TERMINO INDEFINIDO,NaN,2021-09-24,NaN,NaN
1284,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,CHICLAYO,JZULOETA3,ACTIVO,NaN,NaN,TERMINO INDEFINIDO,NaN,2024-09-04,NaN,NaN
1285,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,CHICLAYO,WZULOETA9,ACTIVO,NaN,NaN,TERMINO INDEFINIDO,NaN,2025-01-09,NaN,NaN
1286,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,ILLIMO,JZUÑIGA7,ACTIVO,NaN,NaN,TERMINO INDEFINIDO,NaN,2021-09-28,NaN,NaN


In [13]:
df.to_csv(landing_dir + 'informacion_rrhh.csv', index=False)

## 5. Consolidando los archivos de las solicitudes recibidas según el histórico extraído:

In [21]:
archivos_solicitudes = glob.glob(raw_dir + 'solicitudes' + '*/*')
archivos_solicitudes

['C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw\\solicitudes\\solicitudes_jsdiaz_2025.csv',
 'C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw\\solicitudes\\solicitudes_jsdiaz_2026.csv',
 'C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw\\solicitudes\\solicitudes_jzamorano_2024.csv',
 'C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw\\solicitudes\\solicitudes_jzamorano_2025.csv',
 'C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw\\solicitudes\\solicitudes_mrodriguezpe_2024.csv',
 'C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw\\solicitudes\\solicitudes_mrodriguezpe_2025.csv',
 'C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw\\solicitudes\\solicitudes_mrodriguezpe_2026.csv']

In [22]:
datos_solicitudes = []
for archivo in sorted(archivos_solicitudes):
    datos_solicitudes.append(pd.read_csv(archivo))

df_solicitud = pd.concat(datos_solicitudes, ignore_index=True)
df_solicitud

,Número,Fecha de entrega,Fecha de cierre,Prioridad,Estado,Título,Cerrado por,Nombre de Aplicación,Categoría del sitio,Bdo Tipo de Solicitud,Tipo solici,Tipo de Solicitud
0,RF699325,29/04/25 10:24:20,02/05/25 11:58:20,4 - Baja,Cerrado,Desbloqueo de aplicativo,JSDIAZ,FIRST DATA,A,NaN,NaN,NaN
1,RF699520,29/04/25 12:33:42,02/05/25 09:18:45,4 - Baja,Cerrado,ASIGNACION DE CLAVE MAILPOINT,JSDIAZ,MAILPOINT,A,NaN,NaN,NaN
2,RF699819,29/04/25 16:53:16,06/05/25 14:49:45,4 - Baja,Cerrado,Solicitud de creación de usuario FirstData - F...,JSDIAZ,FIRST VISION,A,NaN,NaN,NaN
3,RF700695,02/05/25 08:56:25,05/05/25 08:40:49,4 - Baja,Cerrado,DESBLOQUEO ACCESO,JSDIAZ,FIRST VISION,A,NaN,NaN,NaN
4,RF700696,02/05/25 08:56:28,02/05/25 11:54:47,4 - Baja,Cerrado,DESBLOQUEO APLICATIVO FD - UGR,JSDIAZ,FIRST DATA,A,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
5882,RF872307,26/05/26 09:32:41,26/05/26 16:21:16,4 - Baja,Cerrado,DESBLOQUEAR USUARIO DE SWIFT 00951,MRODRIGUEZPE,SWIFT DERECHO,A,NaN,NaN,NaN
5883,RF878267,09/06/26 18:08:52,17/06/26 11:35:34,4 - Baja,Cerrado,DOCIT - DESBLOQUEAR USUARIO SWIFT,IAVILA,SWIFT DERECHO,A,NaN,NaN,NaN
5884,RF878819,10/06/26 16:01:26,07/07/26 12:35:17,4 - Baja,Cerrado,Modificacion de perfil,MRODRIGUEZPE,SWIFT DERECHO,A,NaN,NaN,NaN
5885,RF878901,10/06/26 17:30:24,17/06/26 11:39:04,4 - Baja,Cerrado,Reset del Token Usuario de Swift : OCCICCD4,IAVILA,SWIFT IZQUIERDO,A,NaN,NaN,NaN


In [23]:
df_solicitud.to_csv(landing_dir + 'historico_solicitudes.csv', index=False)

## 6. Consolidando los reportes de la aplicación según el histórico extraído:

In [17]:
archivos_apps = glob.glob(raw_dir + 'aplicaciones' + '*/*')
archivos_apps

['C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw\\aplicaciones\\report_users_profile_accesclean_2026_05.csv',
 'C:\\Users\\Juan S. Diaz\\AccesClean_proyecto\\src/../data/raw\\aplicaciones\\report_users_profile_accesclean_2026_06.csv']

In [18]:
datos_apps = []
for archivo in sorted(archivos_apps):
    datos_apps.append(pd.read_csv(archivo))

df_app = pd.concat(datos_apps, ignore_index=True)
df_app

,ID_USER,IDENTIFICATION,NAME_USER,STATE_USER,ORGANIZATION,DATE_CREATE,DATE_LOGIN,ASSIGNED_ROLE
0,PC205JLO,4926,JUANA LOAYZA OLIVA,ACTIVE,205-BANCO_TU_PLATA,2019-10-29 18:01:58,2026-05-16 23:28:15,ADMINISTRADOR
1,PC205AML,13124,ANA MARIA LOCONI QUICIO,ACTIVE,205-BANCO_TU_PLATA,2022-01-14 01:44:58,2026-05-17 05:49:25,ADMINISTRADOR
2,PC205JPF,44442,JEAN PIERRE FLORES REYES,INACTIVE,205-BANCO_TU_PLATA,2021-05-13 12:43:51,2021-11-29 09:41:23,ADMINISTRADOR
3,PC205GMG,20263,GRECIA MARIBEL GONZALES MANAY,ACTIVE,205-BANCO_TU_PLATA,2026-02-27 11:06:02,2026-05-15 11:20:21,ADMINISTRADOR
4,PC205JCG,79883,JUAN CARLOS GONZALEZ LACHOS,ACTIVE,205-BANCO_TU_PLATA,2021-11-05 14:27:07,2026-05-28 13:07:25,ADMINISTRADOR
...,...,...,...,...,...,...,...,...
244,PC205DRH,97646384,Diego Rodriguez Herrera,ACTIVE,205-BANCO_TU_PLATA,2026-05-05 15:00:00,2026-06-24 17:00:00,SOPORTE_TI
245,PC205MDM,89499700,Miguel Diaz Martinez,ACTIVE,205-BANCO_TU_PLATA,2026-05-20 11:00:00,2026-06-25 08:00:00,APROBADOR GENERAL
246,PC205JGM,45871271,Jorge Garcia Morales,ACTIVE,205-BANCO_TU_PLATA,2026-06-21 17:00:00,2026-06-21 17:00:00,APROBADOR GENERAL
247,PC205DMM,71526443,Diego Martinez Martinez,ACTIVE,205-BANCO_TU_PLATA,2026-06-03 14:00:00,2026-06-17 07:00:00,LANZADOR COMPRAS


In [19]:
df_app.to_csv(landing_dir + 'reporte_usuarios_historico.csv', index=False)